# ST-GoG: Spatiotemporal Graph-of-Graphs Framework for Multivariate Time Series Classification

Welcome to the interactive onboarding notebook for **ST-GoG**! This notebook walks you step-by-step through:
1. **Architecture Foundations**: How ResTCN, MultiScaleGCN, and Hierarchical GoG capture cross-channel and multiscale temporal dynamics.
2. **Synthetic / UEA Data Loading**: Creating normalized Multivariate Time Series (MTS) datasets.
3. **Model Initialization**: Configuring the 4-pathway architecture and inspection hooks.
4. **Training with Regularization**: MixUp data augmentation, Stochastic Depth, and Cosine Annealing.
5. **Model Evaluation & Calibration**: Computing Top-1 Accuracy, Macro-F1, and Expected Calibration Error (ECE).
6. **Explainability & Attribution**: Generating Grad*Input saliency heatmaps, sensor ranking, and pathway weight breakdowns.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from stgog import (
    STGoGModel,
    generate_synthetic_mts,
    create_dataloaders,
    STGoGTrainer,
    compute_classification_metrics,
    STGoGExplainer,
)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')


## 1. Data Preparation
Generate a synthetic multivariate time series dataset simulating sensor telemetry with distinct class frequencies and channel interactions.

In [ ]:
X_train, y_train, X_test, y_test, num_classes = generate_synthetic_mts(
    n_train=120,
    n_test=40,
    seq_len=60,
    n_channels=6,
    num_classes=3,
    random_state=42,
)

print(f'Training set shape: {X_train.shape} (N, T, C)')
print(f'Test set shape:     {X_test.shape}')
print(f'Number of classes:   {num_classes}')

train_loader, test_loader = create_dataloaders(
    X_train, y_train, X_test, y_test, batch_size=16, shuffle=True
)


## 2. Model Initialization & Structural Inspection
Instantiate `STGoGModel` with hybrid channel adjacency enabled to capture both physical sensor sequences and dynamic data-driven correlations.

In [ ]:
model = STGoGModel(
    in_channels=6,
    seq_len=60,
    num_classes=num_classes,
    tcn_hidden=32,
    gcn_hidden=32,
    gog_hidden=32,
    num_segments=4,
    gcn_radii=[1, 3, 5],
    use_hybrid_adj=True,
    dropout=0.2,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
print('Initial Pathway Weights:', model.get_pathway_weights())
print('Initial GCN Branch Weights:', model.get_gcn_branch_weights())


## 3. Training with STGoGTrainer
Train using the unified pipeline with Cosine Annealing Warm Restarts, MixUp augmentation, and Snapshot Model Checkpointing.

In [ ]:
trainer = STGoGTrainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    num_classes=num_classes,
    lr=1e-3,
    weight_decay=1e-4,
    mixup_alpha=0.2,
    snapshot_M=2,
    tta_passes=2,
    device=device,
)

results = trainer.fit(epochs=10, print_every=2, verbose=True)
print(f'Best Single Model Accuracy: {results["single_best_acc"]*100:.2f}%')
print(f'Snapshot Ensemble Accuracy: {results["ensemble_acc"]*100:.2f}%')


## 4. Evaluation Metrics & Calibration
Compute comprehensive classification performance including Macro-F1 and Expected Calibration Error (ECE).

In [ ]:
metrics = results['metrics']
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'{k:25s}: {v:.4f}')


## 5. Spatiotemporal Explainability & Attribution
Use `STGoGExplainer` to perform input saliency attribution across both the time dimension and sensor channels, and visualize the 4-panel diagnostic dashboard.

In [ ]:
explainer = STGoGExplainer(model)

# Select a test sample
sample_x = torch.tensor(X_test[0:1], dtype=torch.float32).to(device)
explanation = explainer.compute_saliency(sample_x)

print(f'Predicted Class: {explanation["predicted_class"]} (Confidence: {explanation["class_probability"]*100:.1f}%)')
print('Pathway Fusion Softmax Breakdown:', explanation['pathway_weights'])

summary = explainer.get_attribution_summary(sample_x, top_k_channels=3)
print('Top Most Informative Channels:')
for ch in summary['top_channels']:
    print(f'  Channel {ch["channel_index"]}: {ch["importance"]*100:.2f}% attribution share')

# Render 4-Panel Diagnosis Dashboard
fig = explainer.plot_explanation(
    sample_x,
    explanation=explanation,
    save_path='outputs/notebook_explanation.png'
)
plt.show()
